# A composition atlas: building cell and tissue models by coupling simulators in every direction

_Investigation `composition-atlas` — coder reproduction notebook._

**Question.** This workspace has three tissue-scale simulators (Cellular Potts, vertex, center-based) and three sub-cellular simulators (SBML-ODE, Smoldyn particles, MaBoSS Boolean). How many genuinely different models can we build by COMPOSING them — coupling sub-cellular simulators to each other inside one cell, coupling sub-cellular to tissue, and coupling all scales at once — and what does each composition let us model that no single simulator can alone?

A curated showcase of composition across modelling paradigms, built on the semantics⊥mechanics split (a model's chemistry/logic is separable from the solver that runs it, and the split recurs at every scale). The CoupledCell adapter makes a cell whose intracellular model is itself a coupled composite of sub-models — ODE, particle and Boolean — wired by explicit coupling rules (mode A: a product threshold pins a Boolean input; mode B: a particle count sets an ODE rate; mode C: a Boolean output resets a reaction). Four composites exercise these: a metabolism↔fate oscillator, a particle→ODE catalysis, an all-three-formalisms cell, and the full-stack fateful tissue where fate drives division.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint').is_dir():
    REPO = Path('/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_multicell_blueprint.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Coupled cell: an ODE metabolism and a MaBoSS fate network form a metabolism↔fate oscillator (`coupled-fate-oscillator`)

**Question.** Can an SBML-ODE metabolism and a MaBoSS Boolean fate network be coupled inside one cell, in both directions, to produce a metabolism<->fate feedback oscillator that neither formalism shows alone?

**Claim.** Two sub-cellular simulators of different formalisms — an SBML-ODE metabolism and a MaBoSS Boolean fate network — can be coupled inside one cell in both directions (the ODE's metabolite C pins the fate 'high_C' input, mode A; the fate 'divide' output resets the metabolism, mode C) to produce a metabolism↔fate feedback oscillator that neither formalism exhibits alone.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator` | `viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator` | 40 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `n_cells=1`, `mechanics=vertex`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator`**

One cell, two coupled sub-cellular sims: an SBML-ODE metabolism and a MaBoSS Boolean fate network form a metabolism↔fate feedback oscillator (C builds → pins high_C → divide fires → division resets metabolism → repeat). Runs a small tissue of these coupled cells.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator = {
    'n_cells'  : 4,   # number of coupled cells in the tissue
    'mechanics': 'vertex',   # tissue mechanics backend: 'vertex', 'cpm' or 'munk'
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupled-fate-oscillator ===
STUDY = 'coupled-fate-oscillator'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator = 40
INTERVAL_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator if 'ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator = materialize_composite('viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator)
        print('ran', 'viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator', f'(viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator) for', STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_fate_oscillator, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.coupled_subcellular.coupled_fate_oscillator not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupled-fate-oscillator — behaviour over the run**


In [ ]:
# coupled-fate-oscillator — behaviour over the run
show_viz(_render_one('html:coupled-fate-oscillator.html', {'title': "Metabolite C and fate 'divide' over the run"}, RUNS_DB, STUDY_YAML))

**coupled-fate-oscillator — verdict scorecard**


In [ ]:
# coupled-fate-oscillator — verdict scorecard
show_viz(_render_one('html:coupled-fate-oscillator-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fate-oscillates _(primary)_ | kind=derived_scalar path=atlas_fate_transitions | op >= value 2 |


## Study: Coupled cell: a Smoldyn particle population catalyses an ODE (particle count sets the ODE rate) (`coupled-particle-catalysis`)

**Question.** Can a Smoldyn particle population drive an ODE's kinetics inside one cell, so the ODE product accelerates as the particle count builds (mode B)?

**Claim.** A Smoldyn particle simulator and an SBML-ODE can be coupled inside one cell so that the particle population drives the ODE's kinetics (mode B): the live count of a particle species sets the ODE's rate constant each step, so the ODE product accumulates faster as the particle population builds — a cross-formalism catalysis no single engine expresses.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis` | `viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis` | 30 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `n_cells=1`, `mechanics=vertex`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis`**

One cell, two coupled sub-cellular sims: a Smoldyn particle population (A+B→C, confined to the cell polygon) catalyses an SBML-ODE reaction S→P — the live particle C count sets the ODE rate constant each step (mode B), so P accumulates faster as the particle population builds. Runs a small tissue of these coupled cells.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis = {
    'n_cells'  : 3,   # number of coupled cells in the tissue
    'mechanics': 'vertex',   # tissue mechanics backend: 'vertex', 'cpm' or 'munk'
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupled-particle-catalysis ===
STUDY = 'coupled-particle-catalysis'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis = 30
INTERVAL_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis if 'ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis = materialize_composite('viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis)
        print('ran', 'viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis', f'(viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis) for', STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_particle_catalysis, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.coupled_subcellular.coupled_particle_catalysis not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupled-particle-catalysis — behaviour over the run**


In [ ]:
# coupled-particle-catalysis — behaviour over the run
show_viz(_render_one('html:coupled-particle-catalysis.html', {'title': 'ODE product P (coupled vs fixed-rate control) and particle count'}, RUNS_DB, STUDY_YAML))

**coupled-particle-catalysis — verdict scorecard**


In [ ]:
# coupled-particle-catalysis — verdict scorecard
show_viz(_render_one('html:coupled-particle-catalysis-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| catalysed-product-accumulates _(primary)_ | kind=derived_scalar path=atlas_product_final | op > value 0 |
| particle-driver-builds _(secondary)_ | kind=derived_scalar path=atlas_signal_growth | op > value 0 |


## Study: Coupled cell: all three sub-cellular formalisms and all three coupling modes in one cell (`coupled-triple-cell`)

**Question.** Can one cell run all three sub-cellular formalisms at once, coupled by all three modes (particle->metabolism->fate->reset)?

**Claim.** A single cell can run all three sub-cellular formalisms at once — a Smoldyn particle sim, an SBML-ODE metabolism and a MaBoSS fate network — coupled by all three modes simultaneously: the particle count catalyses the metabolism (B), the metabolite pins the fate input (A), and the fate output resets the metabolism (C).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell` | `viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell` | 36 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `n_cells=1`, `mechanics=vertex`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell`**

One cell running ALL THREE sub-cellular formalisms, coupled by all three modes: a Smoldyn particle population catalyses an SBML-ODE metabolism (mode B), whose metabolite C pins a MaBoSS fate input (mode A), whose 'divide' output resets the metabolism (mode C). Particles → metabolism → fate → division, in a single cell.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell = {
    'n_cells'  : 2,   # number of triple-coupled cells in the tissue
    'mechanics': 'vertex',   # tissue mechanics backend: 'vertex', 'cpm' or 'munk'
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupled-triple-cell ===
STUDY = 'coupled-triple-cell'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell = 36
INTERVAL_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell if 'ov_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell = materialize_composite('viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell)
        print('ran', 'viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell', f'(viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell) for', STEPS_viva_multicell_blueprint_composites_coupled_subcellular_coupled_triple_cell, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.coupled_subcellular.coupled_triple_cell not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupled-triple-cell — behaviour over the run**


In [ ]:
# coupled-triple-cell — behaviour over the run
show_viz(_render_one('html:coupled-triple-cell.html', {'title': "Particle C, metabolite C and fate 'divide' together"}, RUNS_DB, STUDY_YAML))

**coupled-triple-cell — verdict scorecard**


In [ ]:
# coupled-triple-cell — verdict scorecard
show_viz(_render_one('html:coupled-triple-cell-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| triple-chain-oscillates _(primary)_ | kind=derived_scalar path=atlas_fate_transitions | op >= value 2 |


## Study: Fateful tissue: a dividing tyssue whose divisions are decided by each cell's own metabolism→fate chain (`fateful-tissue`)

**Question.** Can a dividing tissue's divisions be decided by each cell's own metabolism->fate chain, with confined particles that split on division — composing every scale at once?

**Claim.** Every scale this workspace can simulate composes into one system: a tyssue of dividing cells, each running a Smoldyn particle sim confined to its moving polygon (split on division) AND an ODE metabolism whose metabolite pins a MaBoSS fate network — and the fate 'divide' output DRIVES the tissue division. Cells grow as they metabolise (target area tracks the metabolite), divide when their own fate fires, and daughters reset and regrow, so the tissue sustains. Biology decides when the tissue divides.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue` | `viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue` | 22 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `seed_per_cell=40`, `c_threshold=14.0`, `max_divisions_per_step=1`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue`**

The full-stack crown: a dividing tyssue where each cell runs a Smoldyn particle sim (split on division) AND an ODE metabolism whose metabolite pins a MaBoSS fate network — and the fate network's 'divide' output drives the tissue division. Biology decides when the tissue divides. Writes a fate-coloured flipbook + GIF to reports/assets/<overlay_out>.html.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue = {
    'blueprint'             : '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/bounded_particles_patch.yaml',   # a path to a YAML file, inline YAML text, or a dict
    'seed_per_cell'         : 50,   # molecules (A+B) seeded per cell
    'c_threshold'           : 14.0,   # metabolite C level that pins the fate 'high_C' input
    'max_divisions_per_step': 1,   # cap on fate-driven divisions per step (tissue stability)
    'overlay_out'           : 'fateful_tissue',   # fate-coloured overlay written to reports/assets/<overlay_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fateful-tissue ===
STUDY = 'fateful-tissue'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue = 22
INTERVAL_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue if 'ov_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue = materialize_composite('viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue)
        print('ran', 'viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue', f'(viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue) for', STEPS_viva_multicell_blueprint_composites_fateful_tissue_fateful_tissue, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.fateful_tissue.fateful_tissue not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fateful-tissue — behaviour over the run**


In [ ]:
# fateful-tissue — behaviour over the run
show_viz(_render_one('html:fateful-tissue.html', {'title': 'Fate-coloured flipbook of the dividing tissue'}, RUNS_DB, STUDY_YAML))

**fateful-tissue — verdict scorecard**


In [ ]:
# fateful-tissue — verdict scorecard
show_viz(_render_one('html:fateful-tissue-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fate-drives-division _(primary)_ | kind=derived_scalar path=atlas_divisions_final | op >= value 1 |
| tissue-sustains _(primary)_ | kind=derived_scalar path=atlas_cells_min | op >= value 4 |


## Study: Smoldyn particles bounded by Cellular Potts (CPM) voxel cells: confinement, conservation across division, reaction equilibrium (`bounded-particles-cpm`)

**Question.** Can a Smoldyn particle sim be confined inside each Cellular Potts (CPM) cell's moving boundary, split into the two daughters on division, and run A+B<->C to equilibrium inside a living tissue?

**Claim.** A Smoldyn particle sim composed with the cpm backend keeps every cell's molecules strictly inside that cell's moving boundary (voxel-region outlines traced from the occupancy), conserves molecule count when a cell divides (each molecule inherited by exactly one daughter, by containment), and lets the reversible reaction A+B<->C reach equilibrium inside the living tissue.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.bounded_particles.bounded_particles` | `viva_multicell_blueprint.composites.bounded_particles.bounded_particles` | 20 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `backend=cpm`, `seed_per_cell=60`, `divide_every=8`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.bounded_particles.bounded_particles`**

Smoldyn particle sims confined inside tyssue cells: each vertex cell is a moving polygon with its own reflective Smoldyn sim (A+B<->C); molecules split into daughters on division. Writes a cell-coloured overlay to reports/assets/<overlay_out>.html.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = {
    'blueprint'    : '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/bounded_particles_patch.yaml',   # a path to a YAML file, inline YAML text, or a dict
    'backend'      : 'vertex',   # multicellular substrate the particles are bounded by: 'cpm', 'vertex', or 'munk'
    'seed_per_cell': 60,   # molecules (A+B) seeded per cell
    'divide_every' : 8,   # divide one cell every N steps (0 = never)
    'overlay_out'  : 'bounded_particles',   # overlay written to reports/assets/<overlay_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bounded-particles-cpm ===
STUDY = 'bounded-particles-cpm'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = 20
INTERVAL_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles if 'ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = materialize_composite('viva_multicell_blueprint.composites.bounded_particles.bounded_particles', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_bounded_particles_bounded_particles['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles)
        print('ran', 'viva_multicell_blueprint.composites.bounded_particles.bounded_particles', f'(viva_multicell_blueprint.composites.bounded_particles.bounded_particles) for', STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.bounded_particles.bounded_particles not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**particles bounded by cpm cells — flipbook**


In [ ]:
# particles bounded by cpm cells — flipbook
show_viz(_render_one('html:bounded-particles-flipbook.html', {'title': 'Smoldyn particles confined in cpm cells, split on division'}, RUNS_DB, STUDY_YAML))

**bounded-particles-cpm — verdict scorecard**


In [ ]:
# bounded-particles-cpm — verdict scorecard
show_viz(_render_one('html:bounded-particles-cpm-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| particles-stay-confined _(primary)_ | kind=derived_scalar path=atlas_confined_min | op >= value 0.9 |
| cells-divide-and-split _(primary)_ | kind=derived_scalar path=atlas_divisions_final | op >= value 1 |


## Study: Smoldyn particles bounded by tyssue (vertex) polygons: confinement, conservation across division, reaction equilibrium (`bounded-particles-vertex`)

**Question.** Can a Smoldyn particle sim be confined inside each tyssue (vertex) cell's moving boundary, split into the two daughters on division, and run A+B<->C to equilibrium inside a living tissue?

**Claim.** A Smoldyn particle sim composed with the vertex backend keeps every cell's molecules strictly inside that cell's moving boundary (junction-tension polygons), conserves molecule count when a cell divides (each molecule inherited by exactly one daughter, by containment), and lets the reversible reaction A+B<->C reach equilibrium inside the living tissue.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.bounded_particles.bounded_particles` | `viva_multicell_blueprint.composites.bounded_particles.bounded_particles` | 20 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `backend=vertex`, `seed_per_cell=60`, `divide_every=8`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.bounded_particles.bounded_particles`**

Smoldyn particle sims confined inside tyssue cells: each vertex cell is a moving polygon with its own reflective Smoldyn sim (A+B<->C); molecules split into daughters on division. Writes a cell-coloured overlay to reports/assets/<overlay_out>.html.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = {
    'blueprint'    : '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/bounded_particles_patch.yaml',   # a path to a YAML file, inline YAML text, or a dict
    'backend'      : 'vertex',   # multicellular substrate the particles are bounded by: 'cpm', 'vertex', or 'munk'
    'seed_per_cell': 60,   # molecules (A+B) seeded per cell
    'divide_every' : 8,   # divide one cell every N steps (0 = never)
    'overlay_out'  : 'bounded_particles',   # overlay written to reports/assets/<overlay_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bounded-particles-vertex ===
STUDY = 'bounded-particles-vertex'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = 20
INTERVAL_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles if 'ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = materialize_composite('viva_multicell_blueprint.composites.bounded_particles.bounded_particles', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_bounded_particles_bounded_particles['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles)
        print('ran', 'viva_multicell_blueprint.composites.bounded_particles.bounded_particles', f'(viva_multicell_blueprint.composites.bounded_particles.bounded_particles) for', STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.bounded_particles.bounded_particles not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**particles bounded by vertex cells — flipbook**


In [ ]:
# particles bounded by vertex cells — flipbook
show_viz(_render_one('html:bounded-particles-flipbook.html', {'title': 'Smoldyn particles confined in vertex cells, split on division'}, RUNS_DB, STUDY_YAML))

**bounded-particles-vertex — verdict scorecard**


In [ ]:
# bounded-particles-vertex — verdict scorecard
show_viz(_render_one('html:bounded-particles-vertex-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| particles-stay-confined _(primary)_ | kind=derived_scalar path=atlas_confined_min | op >= value 0.9 |
| cells-divide-and-split _(primary)_ | kind=derived_scalar path=atlas_divisions_final | op >= value 1 |


## Study: Smoldyn particles bounded by center-based (viva-munk) discs: confinement, conservation across division, reaction equilibrium (`bounded-particles-munk`)

**Question.** Can a Smoldyn particle sim be confined inside each center-based (viva-munk) cell's moving boundary, split into the two daughters on division, and run A+B<->C to equilibrium inside a living tissue?

**Claim.** A Smoldyn particle sim composed with the munk backend keeps every cell's molecules strictly inside that cell's moving boundary (pymunk discs (oblong under crowding)), conserves molecule count when a cell divides (each molecule inherited by exactly one daughter, by containment), and lets the reversible reaction A+B<->C reach equilibrium inside the living tissue.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.bounded_particles.bounded_particles` | `viva_multicell_blueprint.composites.bounded_particles.bounded_particles` | 20 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `backend=munk`, `seed_per_cell=60`, `divide_every=8`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.bounded_particles.bounded_particles`**

Smoldyn particle sims confined inside tyssue cells: each vertex cell is a moving polygon with its own reflective Smoldyn sim (A+B<->C); molecules split into daughters on division. Writes a cell-coloured overlay to reports/assets/<overlay_out>.html.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = {
    'blueprint'    : '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/bounded_particles_patch.yaml',   # a path to a YAML file, inline YAML text, or a dict
    'backend'      : 'vertex',   # multicellular substrate the particles are bounded by: 'cpm', 'vertex', or 'munk'
    'seed_per_cell': 60,   # molecules (A+B) seeded per cell
    'divide_every' : 8,   # divide one cell every N steps (0 = never)
    'overlay_out'  : 'bounded_particles',   # overlay written to reports/assets/<overlay_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bounded-particles-munk ===
STUDY = 'bounded-particles-munk'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = 20
INTERVAL_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles if 'ov_viva_multicell_blueprint_composites_bounded_particles_bounded_particles' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_bounded_particles_bounded_particles = materialize_composite('viva_multicell_blueprint.composites.bounded_particles.bounded_particles', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_bounded_particles_bounded_particles['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles)
        print('ran', 'viva_multicell_blueprint.composites.bounded_particles.bounded_particles', f'(viva_multicell_blueprint.composites.bounded_particles.bounded_particles) for', STEPS_viva_multicell_blueprint_composites_bounded_particles_bounded_particles, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.bounded_particles.bounded_particles not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**particles bounded by munk cells — flipbook**


In [ ]:
# particles bounded by munk cells — flipbook
show_viz(_render_one('html:bounded-particles-flipbook.html', {'title': 'Smoldyn particles confined in munk cells, split on division'}, RUNS_DB, STUDY_YAML))

**bounded-particles-munk — verdict scorecard**


In [ ]:
# bounded-particles-munk — verdict scorecard
show_viz(_render_one('html:bounded-particles-munk-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| particles-stay-confined _(primary)_ | kind=derived_scalar path=atlas_confined_min | op >= value 0.9 |
| cells-divide-and-split _(primary)_ | kind=derived_scalar path=atlas_divisions_final | op >= value 1 |


## Study: Model-definition format: one scene, three cell types, three formalisms (ODE, Smoldyn particles, MaBoSS) sharing an extracellular field (`scene-model-mixed-formalism`)

**Question.** Does a declarative scene plus a per-type model registry compile to ONE composite where ODE, Smoldyn-particle and MaBoSS cells run together sharing an extracellular field?

**Claim.** A declarative scene (cells + types) bound to a per-type model registry compiles to one composite in which metabolic cells run an SBML-style ODE, signalling cells run Smoldyn particles and fate cells run a MaBoSS Boolean network, all coupled to a shared extracellular field through typed membrane transport.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `viva_multicell_blueprint.composites.scene_model.scene_model` | `viva_multicell_blueprint.composites.scene_model.scene_model` | 12 | — |

Declared parameter sets (`study.yaml` variants):

- **reference** — `mechanics=vertex`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.scene_model.scene_model`**

A scene of cells, each bound by type to a model in its own formalism (SBML-ODE, Smoldyn particles, MaBoSS Boolean) sharing one extracellular field.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_scene_model_scene_model = {
    'scene'      : '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/scenes/mixed_formalism_demo.yaml',   # path to a scene YAML
    'mechanics'  : 'vertex',   # tissue mechanics backend: 'vertex', 'cpm' or 'munk'
    'overlay_out': 'scene_model',   # reserved: overlay name under reports/assets/
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: scene-model-mixed-formalism ===
STUDY = 'scene-model-mixed-formalism'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_viva_multicell_blueprint_composites_scene_model_scene_model = 12
INTERVAL_viva_multicell_blueprint_composites_scene_model_scene_model = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_scene_model_scene_model if 'ov_viva_multicell_blueprint_composites_scene_model_scene_model' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_scene_model_scene_model = materialize_composite('viva_multicell_blueprint.composites.scene_model.scene_model', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_scene_model_scene_model['state']}, core=core)
            _comp.run(STEPS_viva_multicell_blueprint_composites_scene_model_scene_model)
        print('ran', 'viva_multicell_blueprint.composites.scene_model.scene_model', f'(viva_multicell_blueprint.composites.scene_model.scene_model) for', STEPS_viva_multicell_blueprint_composites_scene_model_scene_model, 'steps')
    except Exception as _e:
        print(f"viva_multicell_blueprint.composites.scene_model.scene_model not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**mixed-formalism scene — flipbook**


In [ ]:
# mixed-formalism scene — flipbook
show_viz(_render_one('html:scene-model-flipbook.html', {'title': 'ODE / Smoldyn / MaBoSS cells over a shared field'}, RUNS_DB, STUDY_YAML))

**scene-model-mixed-formalism — verdict scorecard**


In [ ]:
# scene-model-mixed-formalism — verdict scorecard
show_viz(_render_one('html:scene-model-mixed-formalism-verdict.html', {'title': 'Behaviour tests: measured value vs pass threshold (code-graded)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| scene-compiles-to-five-cells _(primary)_ | kind=derived_scalar path=atlas_cells_min | op >= value 5 |
| mixed-formalism-produces-state _(primary)_ | kind=derived_scalar path=atlas_total_molecules_final | op > value 0 |
